# Silver Customers

## Import Helper Functions

In [1]:
from src import *

# while it is allowed to use the import above, it is advised to list out what we have imported

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Load Configs

In [2]:
cfg = load_config()

JOB_NAMES = cfg["spark_jobs"]["jobs"]
CATALOG = cfg["general"]["catalog"]
BRONZE_NAMESPACE = cfg["general"]["namespaces"]["bronze"]
SILVER_NAMESPACE = cfg["general"]["namespaces"]["silver"]
BRONZE_TABLE 
PRIMARY_KEY
SILVER_TABLE

NameError: name 'load_config' is not defined

## Import Libraries and Start Session

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["silver"])
        .getOrCreate()
)

## Read from Bronze

In [4]:
BRONZE_TABLE = "polaris.bronze.customers"
bronze_df = spark.read.format("iceberg").table(BRONZE_TABLE)
bronze_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- created_at: long (nullable = true)
 |-- updated_at: long (nullable = true)
 |-- __op: string (nullable = true)
 |-- __ts_ms: long (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_ingest_ts: timestamp (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- spark_ingest_ts: timestamp (nullable = true)



## Common Function 1: Normalize CDC

In [ ]:
def normalize_cdc(df: DataFrame) -> DataFrame:
    """
    1 normalize CDC operation
    2 rename CDC timestamp
    3 convert CDC timestamp to real timestamp
    """
    return (
        df
        .withColumn("cdc_op",
            F.when(F.col("__op") == "r", F.lit("c"))
             .otherwise(F.col("__op"))
        )       
        .withColumnRenamed("__ts_ms", "cdc_ts_ms")
        .withColumn("cdc_ts", F.to_timestamp(F.col("cdc_ts_ms") / 1000))
    )

## Common Function 2: Normalize Decimal Struct

In [ ]:
from pyspark.sql.types import StructType, BinaryType, IntegerType, DecimalType

def normalize_decimal_structs(df: DataFrame) -> DataFrame:
    """
    Convert Avro-decoded decimal structs to numeric types.
    Looks for columns with schema: StructType(scale:int, value:binary)

    df (DataFrame)
     └ df.schema  → StructType
           └ fields  → list of StructField
                 └ field.name       → column name
                 └ field.dataType  → column type (IntegerType, StringType, StructType, etc)
                       └ if StructType: field.dataType.fields → inner list of StructField    
    """
    # Step 1: collect all decimal struct columns
    decimal_cols = []

    for field in df.schema.fields:
        if not isinstance(field.dataType, StructType):
            continue  # skip non-struct columns

        # get inner field names and types
        inner_fields = {f.name: f.dataType for f in field.dataType.fields}

        # detect decimal struct pattern
        if inner_fields.get("scale") == IntegerType() and inner_fields.get("value") == BinaryType():
            decimal_cols.append(field.name)

    # Step 2: apply normalization for each detected column
    for col_name in decimal_cols:
        df = df.withColumn(
                col_name,
                (F.col(f"{col_name}.value").cast("long") /
                F.pow(10, F.col(f"{col_name}.scale")))
                .cast(DecimalType(18,6))
            )

    return df

## Common Function 3: Drop Kafka Metadata

In [ ]:
def drop_kafka_metadata(df: DataFrame) -> DataFrame:
    kafka_metadata_cols = [col for col in df.columns if col.startswith("kafka_")]
    if kafka_metadata_cols:
        return df.drop(*kafka_metadata_cols)
    return df

## Function: Apply Common Transform

In [ ]:
def apply_common_transform(df: DataFrame) -> DataFrame:
    df = normalize_cdc(df)
    df = normalize_decimal_structs(df) 
    df = drop_kafka_metadata(df)
    return df

## Function: Apply Schema Drift

In [13]:
from pyspark.sql.utils import AnalysisException

def apply_schema_drift(silver_ready_df, silver_table_name):
    """
    Detects table existence, handles additive schema evolution,
    and prevents destructive schema changes.
    """
    # ---- Detect table existence ----
    try:
        silver_existing_df = spark.table(silver_table_name)
    except AnalysisException:
        return False   # table does not exist
    
    # ---- Drift Detection ----
    incoming_cols = set(silver_ready_df.columns)
    existing_cols = set(silver_existing_df.columns)
    
    new_cols = incoming_cols - existing_cols
    missing_cols = existing_cols - incoming_cols
    
    if missing_cols:
        raise Exception(f"Destructive schema change detected: {missing_cols}")
    
    # ---- Additive evolution ----
    for col in new_cols:
        dtype = silver_ready_df.schema[col].dataType.simpleString()
        spark.sql(f"""
            ALTER TABLE {silver_table_name} 
            ADD COLUMN {col} {dtype}
        """)

    return True

## Append only model (Functional DE)
I know many would question this code below so I included the reference.
### Reference:
- [The Data Warehouse Setup No One Taught You](https://blog.dataexpert.io/p/the-data-warehouse-setup-no-one-taught)
- [Functional Data Engineering — a modern paradigm for batch data processing](https://maximebeauchemin.medium.com/functional-data-engineering-a-modern-paradigm-for-batch-data-processing-2327ec32c42a)

In [1]:
silver_existing_df.printSchema()

NameError: name 'silver_existing_df' is not defined